# PT-W4-D4 实验：Capability + Policy 执行三角

对应阅读材料：`PT-W4-D4-升级Capability与Policy.md`

**实验目标**：把毕业作品第四层的三个构件写成可运行的小型结构——
1. `CapabilityCard`：动词 + 对象 + 边界 + Consumes/Produces + 三角锚点
2. 三层 `PolicyCard`：审批路径（业务决策面）/ AI 执行（运行面）/ 委托边界（组织授权面）
3. `SkillDescriptor`：复刻 LangChat ADR-003 的真实字段（`effect_policy` / `required_scopes` / `human_review_gate` / `capability_dependencies`）

然后跑通 A101 场景的 **L4 授权判断 + L5 动作建议** 决策链：Agent 建议"创建 Inspection Task"时，必须能说清——这个操作合法吗（Rule）、谁批准（Policy①）、我有没有权限做（Policy②③）。

所有 ID 均取自真实材料：CRE-CON-024（合同终止）、CRE-LEA-011（退场规则）、MI traceability matrix 的 spec 锚点。

## 1. CapabilityCard：能力是"动词 + 对象 + 边界"，不是功能清单

BCM 能力行是散文；Capability Card 把它拆成 AI 可引用的结构。关键设计：
- `produces_events` 指向 D2 的 Event/Effect（能力是事件的发生器）
- `mi_anchor` 指向 MI 追溯矩阵（实现权威），`maturity` 直接借用 MI 状态词表（spec-defined / implemented / accepted）

In [ ]:
from dataclasses import dataclass, field

@dataclass
class CapabilityCard:
    cap_id: str                 # 如 CRE-C-024a（BCM 能力行升级编号）
    verb_object: str            # 动词 + 对象：终止合同
    owned_by: str               # 归属域（Bounded Context）
    boundary: str               # 跨界切分：哪半归本域、哪半归邻域
    consumes: list              # 输入契约
    produces_events: list       # 产出事件（挂 D2 Lifecycle/Effect）
    mi_anchor: str              # MI 追溯锚点（canonical spec + 实现模块）
    maturity: str               # BCM=已核验 / MI=accepted
    skill_candidates: list      # BCM A 列的 Skill 候选（候选 ≠ 事实）

CAPABILITIES = {
    "CRE-C-024a": CapabilityCard(
        cap_id="CRE-C-024a", verb_object="终止合同", owned_by="02 合同管理",
        boundary="终止与清算触发归 02；应收抹除归 04；铺位变空置归 03",
        consumes=["合同状态", "应收余额(04)"],
        produces_events=["ContractTerminated -> occupancy-effect -> 铺位释放"],
        mi_anchor="lease-contract-management spec -> backend/internal/lease/",
        maturity="accepted",
        skill_candidates=["AI 变更影响测算(候选)"]),
    "CRE-C-OPS-INS": CapabilityCard(
        cap_id="CRE-C-OPS-INS", verb_object="创建退租巡检任务", owned_by="05 运营管理",
        boundary="巡检执行归 05；巡检结果回流 02 作为终止守卫输入",
        consumes=["退租申请", "铺位状态(03)"],
        produces_events=["InspectionTaskCreated", "InspectionCompleted -> 解锁终止守卫"],
        mi_anchor="workflow-approvals spec -> backend/internal/workflow/",
        maturity="accepted",
        skill_candidates=[]),
    "CRE-C-FIN-REL": CapabilityCard(
        cap_id="CRE-C-FIN-REL", verb_object="起草费用减免单", owned_by="04 财务管理",
        boundary="四类清算单据资金流归 04；减免审批走 K2；触发归 02",
        consumes=["应收余额", "合同清算状态(02)"],
        produces_events=["ReliefDraftCreated(草稿态,不写世界)"],
        mi_anchor="billing-and-invoicing spec -> backend/internal/billing/",
        maturity="accepted",
        skill_candidates=["财务账龄预警(候选·产品已列)"]),
}

for c in CAPABILITIES.values():
    print(f"{c.cap_id:16} {c.verb_object:10} [{c.maturity:8}] produces: {c.produces_events[0]}")

## 2. 三层 PolicyCard：同一词，三个决策面

昨天 D3 的分界线：**Rule 的主语是世界（状态满不满足），Policy 的主语是人/角色（谁有权决定）**。今天再拆三层：

| 层 | 决策面 | 回答 |
|---|---|---|
| `approval_path` ① | 业务决策 | 这个操作谁批准、走哪条流（K2/BPM）|
| `ai_execution` ② | Agent 运行 | read_only 还是 conditional_write？要哪些 scope？带不带人审门？|
| `delegation` ③ | 组织授权 | 允不允许委托给数字员工？|

注意 CRE-CON-024 那句"合同终止无需审批、终止申请需审批"——**同一操作的两个阶段是两条 Policy**（024a 申请需审批 / 024b 执行免审批），拆开后 Agent 才知道自己在哪个阶段。

In [ ]:
@dataclass
class PolicyCard:
    policy_id: str
    layer: str                 # approval_path / ai_execution / delegation
    subject: str               # 谁被约束
    action: str                # 对什么操作（指向 CapabilityCard.cap_id）
    decision: str              # 裁决结果
    stage: str = ""            # 操作阶段（同一操作分阶段 = 多条 Policy）
    evidence: str = ""         # 证据链回 BCM 行 / 手册行号

POLICIES = [
    # ① 审批路径层（来自 BCM 02 真实规则）
    PolicyCard("CRE-P-024a", "approval_path", "招商/财务/营运", "CRE-C-024a",
               "需 K2 审批", stage="终止申请",
               evidence="CRE-CON-024 + 海鼎合同手册§终止申请 L552-568"),
    PolicyCard("CRE-P-024b", "approval_path", "审批通过后的执行者", "CRE-C-024a",
               "免审批", stage="终止执行", evidence="CRE-CON-024"),
    PolicyCard("CRE-P-OPS-INS", "approval_path", "运营专员/数字员工", "CRE-C-OPS-INS",
               "免审批（常规任务）", evidence="05 运营管理·巡场例行任务"),
    PolicyCard("CRE-P-FIN-REL", "approval_path", "财务专员", "CRE-C-FIN-REL",
               "减免单生效需 K2 审批；起草免审批", evidence="CRE-CON-025 费用减免单(K2 审批)"),
    # ③ 委托边界层（来自 README §7：数字员工是组合层，Skill 候选≠事实）
    PolicyCard("CRE-P-DLG-1", "delegation", "数字员工", "CRE-C-OPS-INS",
               "允许委托（发起+人审后执行）", evidence="README §7 AI 层级"),
    PolicyCard("CRE-P-DLG-2", "delegation", "数字员工", "CRE-C-024a",
               "允许发起申请；执行须人审放行", evidence="CRE-P-024a/b 分阶段"),
]

for p in POLICIES:
    print(f"{p.policy_id:16} [{p.layer:13}] {p.subject} -> {p.action} :: {p.decision}")

In [ ]:
# ② AI 执行层 = LangChat SkillDescriptor（复刻 ADR-003 验证证据里的真实字段）
# 关键红线：conditional_write 必须带 human_review_gate —— 对应 LangChat 校验器
# _conditional_write_requires_review："需审批"在运行时的物化

@dataclass
class SkillDescriptor:
    skill_id: str
    effect_policy: str = "read_only"            # read_only / conditional_write
    required_scopes: list = field(default_factory=list)
    human_review_gate: bool = False
    capability_dependencies: list = field(default_factory=list)

    def __post_init__(self):
        # 复刻 LangChat descriptor 的两条真实校验：
        assert self.effect_policy in ("read_only", "conditional_write"), f"{self.skill_id}: 非法 effect_policy"
        assert "read_only" not in self.required_scopes, f"{self.skill_id}: scope 里禁止 read_only"
        if self.effect_policy == "conditional_write":
            assert self.human_review_gate, f"{self.skill_id}: conditional_write 必须带 human_review_gate（需审批红线）"
        # 三角校验：capability_dependencies 必须能解析回 CapabilityCard
        for cap in self.capability_dependencies:
            assert cap in CAPABILITIES, f"{self.skill_id}: 悬空能力引用 {cap}"

SKILLS = {
    "ops.inspection.create": SkillDescriptor(
        skill_id="ops.inspection.create", effect_policy="conditional_write",
        required_scopes=["ops:inspection:write"], human_review_gate=True,
        capability_dependencies=["CRE-C-OPS-INS"]),
    "contract.termination.impact": SkillDescriptor(
        skill_id="contract.termination.impact", effect_policy="read_only",
        required_scopes=[], human_review_gate=False,
        capability_dependencies=["CRE-C-024a"]),
    "fin.relief.draft": SkillDescriptor(
        skill_id="fin.relief.draft", effect_policy="conditional_write",
        required_scopes=["fin:relief:draft"], human_review_gate=True,
        capability_dependencies=["CRE-C-FIN-REL"]),
}

for s in SKILLS.values():
    print(f"{s.skill_id:30} {s.effect_policy:18} gate={s.human_review_gate!s:5} deps={s.capability_dependencies}")

# 反向验证：conditional_write 无 gate 会被拒绝（这正是 LangChat 校验器的行为）
try:
    SkillDescriptor(skill_id="bad.skill", effect_policy="conditional_write", human_review_gate=False)
except AssertionError as e:
    print(f"\n[校验器拦截] {e}")

## 3. L4/L5 决策链：从"建议动作"到"带依据的授权判断"

升级前，Agent 建议"创建巡检任务"是 LLM 语感生成的热心话；升级后每个建议动作要过四道闸：

```
① 审批路径（谁批准）→ ③ 委托边界（数字员工能不能碰）→ ②a scope 检查（我有没有权限）→ ②b 人审门（要不要人确认）
```

这就是 A101 场景 L4（Policy 判断）+ L5（动作建议）的走线。

In [ ]:
def authorize_action(skill_id: str, granted_scopes: set, trace: list):
    """L4/L5 授权决策链：一个动作建议 → 四道闸 → 带依据的结论"""
    skill = SKILLS[skill_id]
    cap = CAPABILITIES[skill.capability_dependencies[0]]

    # ① 审批路径（找当前阶段适用的 Policy）
    ap = next((p for p in POLICIES if p.layer == "approval_path" and p.action == cap.cap_id), None)
    trace.append(f"①  审批路径  [{ap.evidence}]: {ap.decision}")

    # ③ 委托边界
    dl = next((p for p in POLICIES if p.layer == "delegation" and p.action == cap.cap_id), None)
    if dl is None:
        return "❌ 该能力未授予数字员工（委托边界拒绝）", trace
    trace.append(f"③  委托边界  [{dl.evidence}]: {dl.decision}")

    # ②a scope 检查（复刻 LangChat eligibility.py: set(required_scopes).issubset(scope_grants)）
    if not set(skill.required_scopes).issubset(granted_scopes):
        return "❌ scope_denied：数字员工未获授权范围", trace
    trace.append(f"②a scope 检查  {skill.required_scopes} ⊆ 已授权 → 通过")

    # ②b 人审门
    if skill.human_review_gate:
        trace.append(f"②b 人审门    conditional_write + human_review_gate → 需人确认后执行")
        verdict = "⚠️ 可代办发起，执行需人确认"
    else:
        trace.append(f"②b 人审门    read_only 无 gate → 可直接执行")
        verdict = "✅ 可直接执行（只读，不写世界）"

    trace.append(f"    事后果    {cap.produces_events[0]}")
    return verdict, trace

# A101 场景：数字员工「招商运营员」，持有巡检写权限
granted = {"ops:inspection:write", "contract:read"}

for skill_id in ["contract.termination.impact", "ops.inspection.create", "fin.relief.draft"]:
    trace = []
    verdict, trace = authorize_action(skill_id, granted, trace)
    print(f"\n▶ 动作建议: {skill_id}")
    [print("   " + t) for t in trace]
    print(f"   ⇒ 结论: {verdict}")

## 4. 全链合演：A101 为什么不能出租 → 怎么办 → 谁说了算

In [ ]:
# 复用 D3 的 Rule 引擎结论（L3）+ 今天的 L4/L5，拼出 Agent 最终回答

world = {  # A101 当前世界状态（D1/D2/D3 的产出）
    "space": "A101", "asset_state": "Active",
    "occupancy": "Terminating(退租中)", "inspection": "未完成", "settlement": "进行中",
}

# L3 规则判断（CRE-R-002/R-003，简化复现）
available = world["asset_state"] == "Active" and world["occupancy"] == "无" and world["inspection"] == "无"
guard = world["inspection"] == "完成" and world["settlement"] == "完成"

print("问：A101 铺位为什么不能出租？\n")
print(f"L1/L2  A101 处于 {world['occupancy']}（D1 Relationship + D2 Lifecycle）")
print(f"L3     CRE-R-002 推导 AvailableForLeasing = {available}；阻断点 = 存在未完成退租流程")
print(f"       CRE-R-003 终止守卫 Inspection={world['inspection']} ∧ 清算={world['settlement']} → 未满足")
print()

verdict, trace = authorize_action("ops.inspection.create", granted, [])
print("L4/L5  建议动作：创建 Inspection Task")
print(f"       授权依据：{trace[0]}")
print(f"       执行约束：{trace[3]}")
print(f"       结论：{verdict}")
print(f"       事后果：{trace[4].strip()}（D2 Event 链）")

print("\n终答：该空间存在未完成退租流程（依据 Rule CRE-R-003）。建议创建 Inspection Task（免审批，")
print("      本人可代办发起、执行需您确认）；完成后合同终止 → occupancy-effect → A101 自动释放。")

## 5. 观察与思考

1. **三层 Policy 各挡一道不同的攻击**：①挡流程违规（有签字吗），②挡越权执行（scope 够吗 / 有人审吗），③挡组织越界（这个能力根本不给 AI 碰）。少任何一层，授权链条就有一个可被绕过的洞。
2. **`capability_dependencies` 的三角校验在 `__post_init__` 里就完成了**——悬空的能力引用（Skill 声明了不存在的 Capability）在构造时即失败，而不是运行到一半才发现。这正是 BCM「Skill 候选 ≠ Skill 事实」红线的机器化表达。
3. **L5 的"事后果"引用的是 D2 的 Event 链**——每个建议动作自带"做完世界会怎样"。四层（Ontology / Lifecycle / Rule / Capability+Policy）各答一问，合起来回答完整业务问题。明天 D5 的数字员工定义，就是把今天的三角按岗位角色打包。